In [ ]:
import json
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [ ]:
!pip install sentence-transformers

In [ ]:
instances = []

with open(
    "instances.jsonl",
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        instances.append(
            json.loads(line)
        )

In [ ]:
truth = []

with open(
    "truth.jsonl",
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        truth.append(
            json.loads(line)
        )

In [ ]:
df_instances = pd.DataFrame(
    instances
)

df_truth = pd.DataFrame(
    truth
)

In [ ]:
df = pd.merge(
    df_instances,
    df_truth,
    on="id"
)

In [ ]:
df["headline"] = df["postText"].apply(
    lambda x: x[0]
)

In [ ]:
df["score"] = df["truthMean"]

In [ ]:
df[
    ["headline",
     "score"]
].head()

,headline,score
0,UK’s response to modern slavery leaving victim...,0.133333
1,this is good,1.000000
2,"The ""forgotten"" Trump roast: Relive his brutal...",0.466667
3,Meet the happiest #dog in the world!,0.933333
4,Tokyo's subway is shut down amid fears over an...,0.000000


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#generate embeddings
X = embedding_model.encode(
    df["headline"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/611 [00:00<?, ?it/s]

In [ ]:
y = df["score"]

In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

In [ ]:
# from sklearn.ensemble import RandomForestRegressor

# model = RandomForestRegressor(
#     n_estimators=200,
#     random_state=42,
#     n_jobs=-1
# )
# model.fit(X_train, y_train)

In [ ]:
model = LinearRegression()

model.fit(
    X_train,
    y_train
)

LinearRegression()

In [ ]:
predictions = model.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 0.1522525568454287
RMSE: 0.19196158983812453
R²  : 0.41412505806205313


In [ ]:
joblib.dump(
    model,
    "clickbait_model_minilm.pkl"
)

['clickbait_model_minilm.pkl']

In [ ]:
from google.colab import files

files.download(
    "clickbait_model_minilm.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
with open(
    "embedding_model.txt",
    "w"
) as f:
    f.write(
        "all-MiniLM-L6-v2"
    )

In [ ]:
joblib.dump(
    model,
    "clickbait_model_minilm.pkl"
)
from google.colab import files

files.download(
    "clickbait_model_minilm.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def score_to_class(score):

    if score < 0.30:
        return 0      # Genuine

    elif score < 0.70:
        return 1      # Possibly Clickbait

    return 2          # Highly Clickbait

In [ ]:
y_test_class = [
    score_to_class(x)
    for x in y_test
]

pred_class = [
    score_to_class(x)
    for x in predictions
]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print(
    "Accuracy :",
    accuracy_score(
        y_test_class,
        pred_class
    )
)

print(
    "Precision:",
    precision_score(
        y_test_class,
        pred_class,
        average="weighted"
    )
)

print(
    "Recall   :",
    recall_score(
        y_test_class,
        pred_class,
        average="weighted"
    )
)

print(
    "F1 Score :",
    f1_score(
        y_test_class,
        pred_class,
        average="weighted"
    )
)

Accuracy : 0.6397134083930399
Precision: 0.666876948414334
Recall   : 0.6397134083930399
F1 Score : 0.6355998097669836


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test_class,
    pred_class
)

print(cm)

[[1542  611    2]
 [ 437  863   36]
 [  27  295   95]]


# New Section